In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from snsphd.viz import phd_style
from util.faded_lines import draw_faded_hline
from util.pcr_loader import PcrLoader

colors, swatches = phd_style(jupyterStyle=True)

# ── Paths and parameters ─────────────────────────────────────────────────────
LIGHT_PATH            = "../data/single_wire_Pa230518/R9C10/pcrcurve_cr_20231012-104823.txt"
DARK_PATH             = "../data/single_wire_Pa230518/R9C10/pcrcurve_cr_20231012-104823_dark.txt"
INTEGRATIONS          = 5      # integrations_per_setting
THRESHOLD_SELECTOR    = 3      # which threshold series (TS3 = third threshold level)
RIGHT_CLIP            = -6     # strip last 6 bias points (approach to switching region)
ARROW                 = False
RIGHT_AXIS_COLOR      = "#3054b8"

# ── Load via PcrLoader (exposes .cleaned dict for direct column access) ───────
light_ds = PcrLoader.from_legacy_text(LIGHT_PATH, integrations_per_setting=INTEGRATIONS)
dark_ds  = PcrLoader.from_legacy_text(DARK_PATH,  integrations_per_setting=INTEGRATIONS)

print("Available threshold values (mV):", light_ds.threshold_values())
print("Using: TS%d" % THRESHOLD_SELECTOR)

def get_times_array(dic):
    """
    Extract per-setting dwell times from unixtime differences.
    Each setting takes ~10 s; there are N_thresh transitions between settings.
    We keep time diffs in 9-20 s range, then take every Nth one
    (N = number of thresholds = 3 for this file).
    """
    arr = dic["unixtime_s"] - np.roll(dic["unixtime_s"], 1)
    cleaned = [v for v in arr if 9 < v < 20]
    n_thresh = len(light_ds.threshold_values())
    bifurcated, i = [], 0
    for v in cleaned:
        i += 1
        if i == n_thresh:
            bifurcated.append(v)
            i = 0
    return np.array(bifurcated)

times_pcr  = get_times_array(light_ds.cleaned)
times_dark = get_times_array(dark_ds.cleaned)

ts_key  = f"counts_TS{THRESHOLD_SELECTOR}"
b_key   = f"bias_TS{THRESHOLD_SELECTOR}"

pcr_counts  = light_ds.cleaned[ts_key][:RIGHT_CLIP]
pcr_bias    = -light_ds.cleaned[b_key][:RIGHT_CLIP]        # negated (file uses negative V)
pcr_rate    = pcr_counts / times_pcr[: (RIGHT_CLIP + 1)]

dark_counts = dark_ds.cleaned[ts_key][:RIGHT_CLIP]
dark_bias   = -dark_ds.cleaned[b_key][:RIGHT_CLIP]
dark_rate   = dark_counts / times_dark[: (RIGHT_CLIP + 1)]
yerr        = np.sqrt(np.clip(dark_counts, 0, None)) / times_dark[: (RIGHT_CLIP + 1)]
yerr_lower  = np.minimum(yerr, np.maximum(dark_rate - 1e-9, 0))


In [ ]:
fig, ax_left = plt.subplots(1, figsize=(7, 5))
ax_right = ax_left.twinx()
ax_left.set_zorder(1)
ax_left.patch.set_visible(False)
ax_right.spines["right"].set_visible(True)
ax_right.spines["right"].set_linewidth(1.9)

colors_local = plt.cm.plasma(np.linspace(0.8, 0.4, 1))

ax_left.set_ylim(-180, 4000)
ax_left.yaxis.set_major_locator(MultipleLocator(1000))
ax_right.set_yscale("log")
ax_right.set_ylim(1e-1, 3.12e1)
ax_left.grid(False)
ax_right.grid(False)

ax_left.plot(pcr_bias, pcr_rate,
    marker="o", linestyle="-", label=r"18$\mu$m",
    color=colors_local[0], markersize=4.0, linewidth=1.8, alpha=0.9)

ax_right.errorbar(dark_bias, dark_rate,
    yerr=np.vstack([yerr_lower, yerr]),
    color=RIGHT_AXIS_COLOR, linestyle="-", marker="none",
    linewidth=3.0, markersize=5, capsize=0, alpha=0.3)
ax_right.plot(dark_bias, dark_rate,
    color=RIGHT_AXIS_COLOR, linestyle="-", label="DCR",
    marker="o", markersize=4, linewidth=2.0)

ax_right.set_xlim(*ax_left.get_xlim())
x0, x1 = ax_left.get_xlim()

ymin_l, ymax_l = ax_left.get_ylim()
for y_lin in ax_left.get_yticks():
    if ymin_l <= y_lin <= ymax_l:
        draw_faded_hline(ax_left, y_lin, x0, x1, n_segments=160,
            base_color="black", alpha_max=0.26, lw=0.7, fade_from="left", zorder=0.1)

ymin_r, ymax_r = ax_right.get_ylim()
min_exp = int(np.floor(np.log10(ymin_r)))
max_exp = int(np.floor(np.log10(ymax_r)))
for k in range(min_exp, max_exp + 1):
    for m in range(2, 10):
        y_minor = m * (10**k)
        if ymin_r <= y_minor <= ymax_r:
            draw_faded_hline(ax_right, y_minor, x0, x1, n_segments=160,
                base_color=RIGHT_AXIS_COLOR, alpha_max=0.25, lw=0.5, fade_from="right", zorder=0.05)
    draw_faded_hline(ax_right, 10**k, x0, x1, n_segments=160,
        base_color=RIGHT_AXIS_COLOR, alpha_max=0.25, lw=0.5, fade_from="right", zorder=0.06)

ax_left.set_xlabel(r"Bias Current ($\mu$A)")
ax_left.set_ylabel("Counts per second (Hz)")
ax_right.set_ylabel("Dark Counts per second (Hz)")
ax_right.spines["right"].set_color(RIGHT_AXIS_COLOR)
ax_right.tick_params(axis="y", colors=RIGHT_AXIS_COLOR)
ax_right.yaxis.label.set_color(RIGHT_AXIS_COLOR)

handles_l, labels_l = ax_left.get_legend_handles_labels()
handles_r, labels_r = ax_right.get_legend_handles_labels()
if handles_l:
    ax_left.legend(handles_l, labels_l, fontsize=11, frameon=False, loc="upper left")
if handles_r:
    ax_right.legend(handles_r, labels_r, fontsize=11, frameon=False, loc="lower right")

if ARROW:
    ax_left.annotate("", xytext=(0.215, 1000), xy=(0.25, 1000),
        arrowprops=dict(arrowstyle="->", lw=3, mutation_scale=30, color="#c2cdea", alpha=1))
    leg_d = ax_right.get_legend()
    if leg_d is not None:
        leg_d.set_loc("lower right")

out_dir = Path("../out/single_wire")
out_dir.mkdir(parents=True, exist_ok=True)
st = "_arrow" if ARROW else ""
fig.savefig(out_dir / f"single_wire_faded_grid{st}.png", dpi=300, bbox_inches="tight")
fig.savefig(out_dir / f"single_wire_faded_grid{st}.pdf", dpi=300, bbox_inches="tight")
plt.show()
